# ACCC Contingency Report Analyzer — Demo

End-to-end walk-through of **parser → analytics → diff** against the bundled
synthetic PSS/E ACCC reports.

Run cells **top-to-bottom**.  Swap your own `.rpt` in the *Optional upload*
cell to analyse a real study.

In [10]:
# ── Colab setup ──────────────────────────────────────────────────────────────
# Fill in your GitHub path, uncomment the three lines, then Run All.
#!git clone https://github.com/LucyXYG/AI-powered-contingency-report-analyzer.git
%cd AI-powered-contingency-report-analyzer
!pip install -q pandas plotly

[Errno 2] No such file or directory: 'AI-powered-contingency-report-analyzer'
/content/AI-powered-contingency-report-analyzer


In [11]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from parsers.psse_accc import parse_to_dataframe
from analytics import add_severity, critical_contingencies, vulnerable_elements, summary
from diff import diff_reports, diff_summary

## Optional — Upload Your Own Report

Run the cell below to drop a real PSS/E ACCC `.rpt` file onto the demo.
Skip it to use the bundled `sample_data/accc_base.rpt` throughout.

In [12]:
 BASE_RPT = 'sample_data/accc_base.rpt'   # default; overwritten when a file is uploaded

# try:
#     from google.colab import files as _colab_files
#     print('Click Choose Files to upload a .rpt — or skip this cell for sample data.')
#     _up = _colab_files.upload()
#     if _up:
#         BASE_RPT = next(iter(_up))
#         print('Using uploaded file:', BASE_RPT)
#     else:
#         print('No file uploaded — using', BASE_RPT)
# except ImportError:
#     print('Not in Colab — using', BASE_RPT)

## 1 · Load

Parse the base-case report, compute severity scores, and display every violation
ranked from most to least severe.

In [13]:
df = parse_to_dataframe(BASE_RPT)
df = add_severity(df)

display_cols = [
    'contingency_label', 'violation_type', 'element_type',
    'from_name', 'to_name', 'loading_pct', 'dev_pct', 'severity',
]
(df[display_cols]
 .sort_values('severity', ascending=False)
 .reset_index(drop=True))

,contingency_label,violation_type,element_type,from_name,to_name,loading_pct,dev_pct,severity
0,L_MAPLE-CEDAR_230_1,thermal,branch,CEDAR 230,BIRCH 230,138.1,NaN,45.720
1,L_CEDAR-BIRCH_230_1,thermal,branch,MAPLE 230,CEDAR 230,115.5,NaN,18.600
2,X_BIRCH-ELMHURST_1,thermal,branch,ELMHURST115,OAKDALE115,113.4,NaN,13.400
3,L_MAPLE-CEDAR_230_1,thermal,branch,MAPLE 230,WILLOW230,109.0,NaN,10.800
4,L_ELM-OAKDALE_115_1,thermal,branch,OAKDALE115,PINE 115,108.8,NaN,8.800
5,X_MAPLE_500/230_1,thermal,xfmr,CEDAR 500,CEDAR 230,105.2,NaN,7.800
6,L_BIRCH-ASPEN_230_1,thermal,branch,ASPEN 230,WILLOW230,105.2,NaN,6.240
7,X_BIRCH-ELMHURST_1,thermal,branch,OAKDALE115,PINE 115,105.7,NaN,5.700
8,L_CEDAR-BIRCH_230_1,thermal,branch,BIRCH 230,ASPEN 230,103.5,NaN,4.200
9,X_BIRCH-ELMHURST_1,voltage,bus,PINE 115,None,NaN,-3.29,3.290


## 2 · KPI Cards

Six headline figures for the loaded report.

In [14]:
kpi = summary(df)

fig = make_subplots(
    rows=1, cols=6,
    specs=[[{'type': 'indicator'} for _ in range(6)]],
)
kpi_items = [
    ('Violations',    kpi['total_violations'],     ''),
    ('Thermal',       kpi['thermal_count'],         ''),
    ('Voltage',       kpi['voltage_count'],         ''),
    ('Worst load',    kpi['worst_loading_pct'],     '%'),
    ('Worst ΔV', kpi['worst_voltage_dev_pct'], '%'),
    ('Contingencies', kpi['n_contingencies'],       ''),
]
for col, (title, val, suffix) in enumerate(kpi_items, start=1):
    fig.add_trace(
        go.Indicator(
            mode='number',
            value=val,
            title={'text': title, 'font': {'size': 13}},
            number={'suffix': suffix, 'font': {'size': 30}},
        ),
        row=1, col=col,
    )
fig.update_layout(height=200, margin=dict(t=30, b=0, l=20, r=20))
fig.show()

## 3 · Critical Contingencies

Horizontal bars ranked by **total severity**; annotated with violation count.
Count-rank and severity-rank can disagree — both perspectives matter.

In [15]:
cc = (critical_contingencies(df, sort_by='total_severity')
      .sort_values('total_severity'))          # ascending so highest is at top

fig = go.Figure(go.Bar(
    y=cc['contingency_label'],
    x=cc['total_severity'],
    orientation='h',
    text=[f'{n} violation' + ('' if n == 1 else 's') for n in cc['violation_count']],
    textposition='outside',
    marker=dict(
        color=cc['total_severity'],
        colorscale='YlOrRd',
        showscale=True,
        colorbar=dict(title='Total<br>severity'),
    ),
))
fig.update_layout(
    title='Critical Contingencies — Total Severity Score',
    xaxis_title='Total Severity Score',
    height=400,
    margin=dict(l=230, r=130, t=60, b=40),
)
fig.show()

## 4 · Vulnerable Elements

Elements that violate under the most contingencies are the strongest
reinforcement candidates — fix one, improve multiple contingency outcomes.

In [16]:
ve = vulnerable_elements(df).copy()
ve['label'] = ve.apply(
    lambda r: (f'{r["from_name"]} → {r["to_name"]}'
               if pd.notna(r['to_name']) else r['from_name']),
    axis=1,
)
ve_plot = ve.sort_values('n_contingencies')     # ascending so highest is at chart top

fig = go.Figure(go.Bar(
    y=ve_plot['label'],
    x=ve_plot['n_contingencies'],
    orientation='h',
    text=[f'max sev {s:.1f}' for s in ve_plot['max_severity']],
    textposition='outside',
    marker=dict(
        color=ve_plot['max_severity'],
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title='Max<br>severity'),
    ),
))
fig.update_layout(
    title='Vulnerable Elements — Reinforcement Candidates',
    xaxis=dict(title='Distinct contingencies triggering a violation', dtick=1),
    height=500,
    margin=dict(l=250, r=160, t=60, b=40),
)
fig.show()

## 5 · Overload Heatmap

Visual centrepiece: **contingency × monitored element**, colour = loading %.
Blank cells = no thermal violation recorded for that pair.

In [17]:
thermal = df[df['violation_type'] == 'thermal'].copy()
thermal['element'] = (
    thermal['from_name'].str.strip() + ' → ' + thermal['to_name'].str.strip()
)
pivot = thermal.pivot_table(
    index='contingency_label',
    columns='element',
    values='loading_pct',
    aggfunc='max',
)
pivot.columns.name = None
pivot.index.name = None

z = pivot.values.astype(float)
text_grid = [
    [f'{v:.1f}%' if not np.isnan(v) else '' for v in row]
    for row in z
]

fig = go.Figure(go.Heatmap(
    z=z,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='YlOrRd',
    zmin=100,
    zmax=max(float(np.nanmax(z)), 110),
    text=text_grid,
    texttemplate='%{text}',
    colorbar=dict(title='Loading %'),
    hoverongaps=False,
))
fig.update_layout(
    title='Thermal Overload Heatmap: Contingency × Element',
    height=420,
    margin=dict(l=230, r=40, b=190, t=60),
    xaxis=dict(tickangle=-45, tickfont=dict(size=10)),
)
fig.show()

## 6 · Before / After Diff

Compare the base-case and post-upgrade reports.
Bars point **left** for resolved violations (gone), **right** for all others.
Bar length = severity score.

In [18]:
POST_RPT = 'sample_data/accc_postupgrade.rpt'
post_df  = parse_to_dataframe(POST_RPT)

ddf = diff_reports(df, post_df).copy()
ds  = diff_summary(ddf)

headline = (
    f"{ds['n_resolved']} resolved  ·  "
    f"{ds['n_new']} new  ·  "
    f"{ds['worsened']} worsened  ·  "
    f"{ds['improved']} improved  ·  "
    f"{ds['unchanged']} unchanged"
    f"   (net: {'+' if ds['net_change'] >= 0 else ''}{ds['net_change']})"
)

STATUS_COLOR = {
    'resolved':  '#2ca02c',   # green
    'improved':  '#1f77b4',   # blue
    'unchanged': '#aec7e8',   # grey-blue
    'worsened':  '#ff7f0e',   # orange
    'new':       '#d62728',   # red
}
STATUS_ORDER = ['resolved', 'improved', 'unchanged', 'worsened', 'new']


def _bar_x(row):
    if row['status'] == 'resolved':
        return -(row['base_severity'] if pd.notna(row['base_severity']) else 0.0)
    return float(row['post_severity'] if pd.notna(row['post_severity']) else 0.0)


ddf['bar_x'] = ddf.apply(_bar_x, axis=1)
ddf['label'] = ddf['contingency_label'] + ' / ' + ddf['element']
ddf['_ord']  = ddf['status'].map({s: i for i, s in enumerate(STATUS_ORDER)})
ddf = ddf.sort_values(['_ord', 'bar_x']).reset_index(drop=True)

fig = go.Figure()
for status in STATUS_ORDER:
    grp = ddf[ddf['status'] == status]
    if grp.empty:
        continue
    fig.add_trace(go.Bar(
        x=grp['bar_x'],
        y=grp['label'],
        name=status,
        orientation='h',
        marker_color=STATUS_COLOR[status],
        width=0.6,
    ))

fig.update_layout(
    title=headline,
    barmode='overlay',
    xaxis=dict(
        title='← resolved   |   severity →',
        zeroline=True, zerolinewidth=2, zerolinecolor='black',
    ),
    yaxis=dict(autorange='reversed', tickfont=dict(size=10)),
    height=620,
    margin=dict(l=385, r=120, t=60, b=40),
    legend=dict(title='Status', traceorder='normal'),
)
fig.show()